# Dr. Saeid Homayoun — AuditBERT US CAM / UK KAM Research Training

**Research infrastructure:** Google Drive (private corpus and checkpoints) · GitHub `Saehon/Saeid-Homayoun` (versioned code) · Hugging Face `SADHON` (private model cards) · ChatGPT/DARWIN (scientific review).

This notebook is a **training entry point, not a report of completed FinBERT weights**. The source Drive reports contain 45,924 US CAM rows and 86,122 Europe-wide KAM rows, with 35,886 **UK-headquartered** candidates, NOT verified UK auditor-report regime. The previous paper's KAM20240505 file stopped at 65,535 rows. The independent private TF–IDF baselines have already been run on source data and logged in the canonical Drive registry.

**Important:** Original KAM notebooks contained HF tokens; revoke/rotate before using any of them. This notebook **does not load or execute the original notebooks or send source text to GitHub or the public Hugging Face Space**. Provide Google authorization in your own Colab login. Colab GPU quota and cost depend on your account; this notebook does not create paid services automatically.

In [ ]:
# Step 1 — Inspect available compute and fetch version-controlled PUBLIC-SAFE code
import subprocess, os, sys
from pathlib import Path
subprocess.run(["git","clone","--single-branch","--branch",
 "research/saeid-auditbert-v2-20261016",
 "https://github.com/Saehon/Saeid-Homayoun.git",
 "/content/saeid-auditbert-repo"],check=True)
ROOT = Path("/content/saeid-auditbert-repo/research/saeid-homayoun-auditbert-v2")
print("Code:",ROOT)
try:
 import torch
 print("Torch CUDA available:",torch.cuda.is_available())
except ImportError:
 print("Torch not installed. Install it before GPU training.")

In [ ]:
# Step 2 — Authorize this Colab runtime to see the EXISTING PRIVATE Google Drive folder
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
BASE = Path("/content/drive/MyDrive/CAMbERT202610")
# Change BASE only if that existing folder is stored elsewhere in your Drive.
CAM = BASE/"CAM.csv"
KAM = BASE/"KAM.csv"
for name,path in (("CAM",CAM),("KAM",KAM)):
 print(name,"exists:",path.is_file(),"bytes:",path.stat().st_size if path.is_file() else "not mounted")
if not CAM.is_file() or not KAM.is_file():
 raise FileNotFoundError("Find the mounted path to the existing CAMbERT202610 folder and update BASE. Do not create duplicate data files.")

## SOURCE REVIEW ADDENDUM: original US audit-opinion date crosswalk

The user-shared [Claude folder](https://drive.google.com/drive/folders/1NMX2_Ojg_maMyokV1j3rlt55aC3KWNV4?usp=sharing) contains the same authoritative CAMbERT202610 archive plus [`PHASE_1_CAM/03_CAM_CLEANED_DATASET/OPINION_DATES.csv`](https://drive.google.com/file/d/1sYJEzNMfHr9DUxVun7ezrvYXEh-gsBtg/view). The original raw `CAM.csv` still has **no direct full audit-report date** and `FISCAL_YE` is an MMDD field, not a year. However, this *separate* Phase 1 file contains audited date-alignment candidates at **audit-opinion level**, and should be checked before calling US CAM forward-time validation impossible.

The source-linked Phase 1 report gives 32,859 opinion keys: `VERIFIED_TEXT_AND_FORM_AP` 19,264; `ALIGNED_FORM_AP_UNIQUE` 2,635; `ALIGNED_AMBIGUOUS` 4,263; `UNDATED` 6,697. Only the first cohort is a candidate **primary** chronological sample; the second is a sensitivity cohort subject to its masked-date error rate; the others remain **UNKNOWN**. There are fewer eligible dated opinions than full CAM records, so selection/coverage must be reported. Neither UK KAM source year nor UK auditor reporting regime is resolved by this US-only date file.

This diagnostic does not change the baseline issuer holdout, *does not automatically merge with ICFR*, and *does not train a model*. A dedicated temporal holdout must be created only after source-date and issuer/year joins are verified and frozen. [Human coding sample](https://drive.google.com/file/d/1b22bfuKXywhCEDUK80yEHu0Svj0Cx9vg/view) exists but its 260 coder fields are currently unfilled. [13-category Phase 1 account codebook](https://drive.google.com/file/d/1UBPhRsHfcYJuPvf8oJ4Zo9l_jnCtK-BK/view) is an automated crosswalk, not expert ground truth. The separate [DARWIN-ACRIS notebook](https://github.com/Saehon/RISK-INDEX/blob/main/notebooks/DARWIN_V9_TAR_FinBERT_Colab.ipynb) is for *optional economic-risk analysis* after classifier training/validation, not a substitute for this benchmark.

In [ ]:
# Read-only audit of existing dated US CAM opinions; aggregate counts only.
# This cell never writes private text, API tokens, audit-opinion IDs or raw rows to GitHub/HF/Kaggle.
from pathlib import Path
import pandas as pd
opinion_dates_path = BASE / "PHASE_1_CAM" / "03_CAM_CLEANED_DATASET" / "OPINION_DATES.csv"
if opinion_dates_path.is_file():
    opinion_dates = pd.read_csv(
        opinion_dates_path,
        dtype={"AUDIT_OPINION_FKEY": "string", "CIK": "string",
               "status": "string", "fiscal_year": "string",
               "report_date": "string", "date_usable": "string"},
        low_memory=False)
    assert len(opinion_dates) > 1000, "Source date crosswalk unexpectedly small"
    assert opinion_dates["AUDIT_OPINION_FKEY"].notna().all()
    assert opinion_dates["AUDIT_OPINION_FKEY"].is_unique, "Duplicated opinion-date keys require manual resolution"
    classes = opinion_dates["status"].value_counts(dropna=False)
    expected = {"VERIFIED_TEXT_AND_FORM_AP", "ALIGNED_FORM_AP_UNIQUE",
                "ALIGNED_AMBIGUOUS", "UNDATED"}
    if not set(classes.index.astype(str)).issubset(expected):
        raise ValueError("Unexpected date provenance class. Re-audit before using years.")
    highconf = opinion_dates["status"].eq("VERIFIED_TEXT_AND_FORM_AP")
    highconf &= opinion_dates["date_usable"].astype(str).str.lower().eq("true")
    assert not opinion_dates.loc[highconf, "report_date"].isna().any()
    print("Existing date provenance classes (OPINIONS; not CAM count):",
          {str(k): int(v) for k, v in classes.items()})
    print("High-confidence candidate opinions with report dates:", int(highconf.sum()))
    # Optional key-coverage inspection using two minimal original CAM identifier columns.
    cam_keys = pd.read_csv(CAM, usecols=["AUDIT_OPINION_FKEY", "COMPANY_FKEY"],
                           dtype="string", low_memory=False)
    linked = cam_keys.merge(
        opinion_dates[["AUDIT_OPINION_FKEY", "status", "date_usable"]],
        on="AUDIT_OPINION_FKEY", how="left", validate="many_to_one",
        indicator=True)
    print("Original CAM matter records matched to date lookup:",
          int(linked["_merge"].eq("both").sum()), "of", len(linked))
    print("CAM matter rows by date provenance:",
          {str(k): int(v) for k, v in linked["status"].value_counts(dropna=False).items()})
    del cam_keys, linked, opinion_dates
else:
    print("Date crosswalk not in current mounted folder. Locate the original PHASE_1_CAM",
          "in your shared Drive folder. No synthetic dates will be generated and",
          "no temporal model claim is permitted.")


## Scientific and security gates

Before GPU execution, verify that you are authorized to train on the text and topic labels, and that any previously exposed Hugging Face tokens embedded in legacy notebooks were revoked/rotated. **Neither US/UK raw CSV contains independently verified audit-report year.** This notebook runs **issuer-disjoint** tests only; never report a time-forward study from these CSVs. UK means headquarters code GB; it **does not establish ISA (UK) reporting jurisdiction**.

The old notebook code is not used or executed. 

In [ ]:
# Step 3 — Manual governance attestations. Do not bypass these gates.
AUTHORIZED_PRIVATE_SOURCE_USE = False   # Set True only after confirming your rights to privately train on source disclosures + labels
OLD_NOTEBOOK_HF_TOKENS_REVOKED = False    # Set True after revoking/rotating the six embedded original HF tokens
RUN_GPU_FINBERT = False                  # Set True to intentionally begin full FinBERT training
print("Use-rights gate:",AUTHORIZED_PRIVATE_SOURCE_USE)
print("Credentials remediation gate:",OLD_NOTEBOOK_HF_TOKENS_REVOKED)
print("GPU training switch:",RUN_GPU_FINBERT)

In [ ]:
# Step 4 — Reproduce source quality and CPU baselines, preserving private data.
# These commands do NOT require source-use attestation for read-only internal analysis,
# but do NOT run them if the source's own access rules prohibit such analysis.
import subprocess
import sys
import pandas as pd
private_runtime = Path("/content/auditbert_usuk_private_runtime")
private_runtime.mkdir(parents=True,exist_ok=True)
preflight = ROOT/"large_corpus_preflight.py"
baseline = ROOT/"real_corpus_tfidf.py"
for label,src in (("us_cam",CAM),("uk_hq_kam",KAM)):
 cmd=[sys.executable,str(preflight),
      "--cam" if label=="us_cam" else "--kam",str(src)]
 subprocess.run(cmd,check=True)
 print("Source verified:",label)
 # Real-data baseline is optional here because its earlier completed metrics
 # are already documented in the project Drive research registry.
REPEAT_BASELINE = False
if REPEAT_BASELINE:
 for label,src in (("us_cam",CAM),("uk_hq_kam",KAM)):
  subprocess.run([sys.executable,str(baseline),
    "--input",str(src),"--source",label,"--output",
    str(private_runtime/(label+"_tfidf.json"))],check=True)

In [ ]:
# Step 5 — Controlled GPU environment: no training until governance attestations are set.
if RUN_GPU_FINBERT:
 if not AUTHORIZED_PRIVATE_SOURCE_USE or not OLD_NOTEBOOK_HF_TOKENS_REVOKED:
  raise PermissionError("Rights and credential-remediation gates must both pass before GPU training.")
 subprocess.run([sys.executable,"-m","pip","install","-q",
    "transformers>=4.48,<5","accelerate>=1,<2",
    "datasets>=3,<5","sentencepiece","scikit-learn","pandas",
    "openpyxl"],check=True)
 import torch
 if not torch.cuda.is_available():
  raise RuntimeError("GPU not available. Select Runtime > Change runtime type > GPU. Do not start the large FinBERT stage on CPU.")
 print("GPU:",torch.cuda.get_device_name(0))
else:
 print("GPU FinBERT NOT started. Set the three manual gates after source and credential verification.")

In [ ]:
# Step 6 — Derive exact SAME cleaned, issuer-disjoint / rare-label protocol
# used in real_corpus_tfidf.py. Create TEMP private normalized input only on
# the Colab runtime (not in public GitHub, Hub or Drive).
import importlib.util
spec=importlib.util.spec_from_file_location("real_corpus_tfidf",str(baseline))
m=importlib.util.module_from_spec(spec)
spec.loader.exec_module(m)
private_inputs={}
for label,path in (("us_cam",CAM),("uk_hq_kam",KAM)):
 d, schema, facts=m.load_clean(path,label)
 tr, va, te=m.group_split(d,schema["issuer"],42)
 frequencies=tr[schema["topic"]].value_counts()
 allowed=set(frequencies[frequencies>=100].index)
 if len(allowed)<2:
  raise ValueError(label+": insufficient topic support")
 d["topic"]=d[schema["topic"]].where(d[schema["topic"]].isin(allowed),other="OTHER_RARE").astype(str)
 d["title"]=d["TITLE"].astype(str)
 d["description"]=d["DESCRIPTION"].astype(str)
 d["company"]=d[schema["issuer"]].astype(str)
 out=private_runtime/(label+"_TEMP_training_only.csv")
 d[["title","description","topic","company"]].to_csv(out,index=False)
 private_inputs[label]=out
 print(label,"rows:",len(d),"classes:",d["topic"].nunique(),
       "train groups:",tr[schema["issuer"]].nunique(),
       "temporary private CSV:",str(out))
# Future enhancement: create permanent split manifests only by hashes, and
# human-report-regime/date crosswalks in existing Drive workbook.

In [ ]:
# Step 7 — FULL eligible US CAM / UK-HEADQUARTERS proxy FinBERT training.
# Each model is trained independently; no cross-jurisdiction mixing.
# New model checkpoints are written into the existing PRIVATE project Drive folder.
from pathlib import Path
checkpoint_base = BASE/"private_auditbert_v2_checkpoints"
if RUN_GPU_FINBERT:
 if not AUTHORIZED_PRIVATE_SOURCE_USE or not OLD_NOTEBOOK_HF_TOKENS_REVOKED:
  raise PermissionError("Governance gates are missing")
 for label in ("us_cam","uk_hq_kam"):
  target=checkpoint_base/label
  cmd=[sys.executable,str(ROOT/"train_finbert_topic.py"),
    "--input",str(private_inputs[label]),"--output",str(target),
    "--base-model","yiyanghkust/finbert-tone",
    "--split","company","--epochs","2","--batch-size","4",
    "--max-length","384","--seed","42"]
  print("Training",label,"with issuer-disjoint holdouts; this is NOT report-year-forward.")
  subprocess.run(cmd,check=True)
  assert (target/"experiment_report.json").is_file(), "No verified output: do not mark PASS"
  print("Saved PRIVATE checkpoint and real metrics in",target)
else:
 print("Full FinBERT training NOT executed. Enable RUN_GPU_FINBERT after gates.")

## Model release and scientific review

**Do not publish checkpoints or new F1 values to Hugging Face until the results are read back and independently reviewed.** The two existing private `SADHON` repositories are documentation-only until weights are trained, data-use rights and model redistribution rights are confirmed, and leakage/error analysis passes. UK HQ is not an adequate proxy for audited ISA (UK) reporting where jurisdiction claims are required.

Additional FT50/AJG4 research gates: verify audit-report date via `AUDIT_OPINION_FKEY`; check repeated audit opinion/firm families and near-identical paragraphs; reconcile historic 48 vs 49 KAM topics and UK/US 59/55 native taxonomies; expert gold labels and confidence intervals; TF–IDF, FinBERT and optional DAPT/joint comparisons on frozen holdouts. Unknown and negative results must be retained.

Original collaborative paper: https://ssrn.com/abstract=7272139

Primary record and experiment status: https://docs.google.com/document/d/1YmktPLQ-be1sIFd-Ghlpvd7r156C2boC3y9eofM8p6k/edit

GitHub source branch and PR: https://github.com/Saehon/Saeid-Homayoun/pull/185